# Non-Grav Support Walkthrough

This notebook demonstrates the current non-gravitational parameter support in `adam_core` and `adam_assist`.

It is organized around real fixtures already checked into the repository:

- `99942` from SBDB: supported `A1/A2` asteroid case
- `54509` from SBDB: supported `A2` asteroid case with fixed Marsden constants
- `101955` from SBDB: unsupported SRP-only case (`AMRAT`, `RHO`)
- `67P` from SBDB: unsupported cometary case with `DT`
- `99942` from NEOCC: supported `A2` handoff with non-estimated `AMRAT = 0`

For the supported cases we propagate with and without non-gravs enabled, compare the trajectories directly, and optionally compare against a live JPL Horizons truth source when network access is available.

## What This Notebook Shows

Current runtime behavior:

- `adam_core.Orbits` can carry first-class non-gravitational parameters.
- SBDB, NEOCC, and MPCQ can populate the canonical `A1/A2/A3` fields.
- Non-grav uncertainties live in the coordinate covariance itself, which extends
  to a 9x9 matrix over `(coordinates, A1, A2, A3)` when the source solution
  includes them (cross-covariances included).
- Source parameters outside `A1/A2/A3` (`DT`, `AMRAT`, `RHO`, Marsden `g(r)`
  constants, ...) are dropped at ingestion with a warning, and their covariance
  dimensions are marginalized out.
- `adam_assist` supports propagation with Marsden-style `A1/A2/A3`.
- `propagate_2body` rejects orbits with non-zero non-gravitational parameter values.

This notebook is meant to be easy to adapt when we want to inspect additional real-world objects.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from adam_assist.propagator import ASSISTPropagator, _extract_assist_particle_params
from adam_core.orbits import Orbits
from adam_core.orbits.query.horizons import _get_horizons_vectors
from adam_core.orbits.query.neocc import _non_gravitational_parameters_from_neocc, _parse_oef
from adam_core.orbits.query.sbdb import _orbits_from_sbdb_payloads
from adam_core.time import Timestamp

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "adam_core").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing src/adam_core")


ROOT = find_repo_root(Path.cwd().resolve())
TESTDATA = ROOT / "src" / "adam_core" / "orbits" / "query" / "tests" / "testdata"
SBDB_DIR = TESTDATA / "sbdb"
NEOCC_DIR = TESTDATA / "neocc"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

propagator = ASSISTPropagator()


In [ ]:
def load_sbdb_orbit(fixture_name: str, object_id: str | None = None) -> Orbits:
    payload = json.loads((SBDB_DIR / fixture_name).read_text())
    if object_id is None:
        object_id = Path(fixture_name).stem.replace("_phys", "")
    return _orbits_from_sbdb_payloads([object_id], [payload])


def load_neocc_nongrav(fixture_name: str):
    data = _parse_oef((NEOCC_DIR / fixture_name).read_text())
    return data, _non_gravitational_parameters_from_neocc(data)


def nongrav_summary(orbits: Orbits) -> pd.DataFrame:
    ng = orbits.non_gravitational_parameters
    return pd.DataFrame(
        {
            "source": ng.source.to_pylist(),
            "A1": ng.A1.to_pylist(),
            "A2": ng.A2.to_pylist(),
            "A3": ng.A3.to_pylist(),
            "has_covariance_block": orbits.coordinates.covariance.nongrav_block_mask(),
        }
    )


def zero_supported_nongrav(orbits: Orbits) -> Orbits:
    ng = orbits.non_gravitational_parameters
    zeroed_kwargs = {"source": ng.source.to_pylist()}
    for field in ["A1", "A2", "A3"]:
        zeroed_kwargs[field] = [
            0.0 if value is not None else None for value in getattr(ng, field).to_pylist()
        ]

    return orbits.set_column(
        "non_gravitational_parameters",
        type(ng).from_kwargs(**zeroed_kwargs),
    )


def future_times(orbits: Orbits, day_offsets: np.ndarray) -> Timestamp:
    base_mjd = orbits.coordinates.time.mjd().to_numpy(zero_copy_only=False)[0]
    return Timestamp.from_mjd(base_mjd + np.asarray(day_offsets, dtype=np.float64), scale=orbits.coordinates.time.scale)


def propagate_pair(orbits: Orbits, day_offsets: np.ndarray) -> tuple[Timestamp, Orbits, Orbits, pd.DataFrame]:
    times = future_times(orbits, day_offsets)
    with_nongrav = propagator.propagate_orbits(orbits, times)
    without_nongrav = propagator.propagate_orbits(zero_supported_nongrav(orbits), times)
    diff = with_nongrav.coordinates.values - without_nongrav.coordinates.values
    summary = pd.DataFrame(
        {
            "days_from_epoch": day_offsets,
            "delta_r_au": np.linalg.norm(diff[:, :3], axis=1),
            "delta_v_au_per_day": np.linalg.norm(diff[:, 3:], axis=1),
        }
    )
    return times, with_nongrav, without_nongrav, summary


def plot_supported_comparison(label: str, summary: pd.DataFrame) -> None:
    display(Markdown(f"**{label} delta summary**"))
    display(
        summary.assign(
            log10_delta_r_au=np.log10(np.clip(summary["delta_r_au"], 1e-300, None)),
            log10_delta_v_au_per_day=np.log10(np.clip(summary["delta_v_au_per_day"], 1e-300, None)),
        )
    )


def try_horizons_vectors(object_id: str, times: Timestamp):
    try:
        return _get_horizons_vectors([object_id], times, location="@sun", id_type="smallbody", aberrations="geometric", refplane="ecliptic")
    except Exception as exc:
        print(f"Horizons comparison skipped for {object_id}: {type(exc).__name__}: {exc}")
        return None


def compare_to_horizons(object_id: str, propagated: Orbits, times: Timestamp) -> pd.DataFrame | None:
    vectors = try_horizons_vectors(object_id, times)
    if vectors is None:
        return None

    horizons_xyz = vectors[["x", "y", "z"]].to_numpy(dtype=float)
    our_xyz = propagated.coordinates.values[:, :3]
    residual = np.linalg.norm(our_xyz - horizons_xyz, axis=1)
    return pd.DataFrame(
        {
            "days_from_epoch": times.mjd().to_numpy(zero_copy_only=False) - times.mjd().to_numpy(zero_copy_only=False)[0],
            "horizons_position_residual_au": residual,
        }
    )


def attempt_supported_case(label: str, object_id: str, orbit: Orbits, day_offsets: np.ndarray) -> None:
    display(Markdown(f"### {label}"))
    display(nongrav_summary(orbit))

    times, with_nongrav, without_nongrav, summary = propagate_pair(orbit, day_offsets)
    display(summary)
    plot_supported_comparison(label, summary)

    with_horizons = compare_to_horizons(object_id, with_nongrav, times)
    without_horizons = compare_to_horizons(object_id, without_nongrav, times)
    if with_horizons is not None and without_horizons is not None:
        comparison = with_horizons.copy()
        comparison["without_nongrav_horizons_residual_au"] = without_horizons["horizons_position_residual_au"]
        display(comparison)
        display(Markdown(f"**{label}: optional JPL Horizons comparison**"))
        display(
            comparison.assign(
                log10_with_nongrav=np.log10(np.clip(comparison["horizons_position_residual_au"], 1e-300, None)),
                log10_without_nongrav=np.log10(np.clip(comparison["without_nongrav_horizons_residual_au"], 1e-300, None)),
            )
        )


## Case 1: SBDB `99942`

This is a real asteroid case with supported estimated parameters (`A1`, `A2`).

We will:

1. Load the orbit and inspect the canonical non-grav fields.
2. Propagate with the fitted `A1/A2` values.
3. Propagate again with `A1/A2` zeroed while keeping the same initial state.
4. Compare the trajectories directly.
5. Optionally compare both runs to live JPL Horizons vectors if network access is available.

In [ ]:
day_offsets = np.array([30.0, 365.0, 730.0, 1825.0, 3650.0])
orbit_99942 = load_sbdb_orbit("99942_phys.json", object_id="99942")
attempt_supported_case("SBDB 99942", "99942", orbit_99942, day_offsets)


## Case 2: SBDB `54509`

This is another supported asteroid case. The fixture includes `A2` plus fixed Marsden constants (`R0`, `ALN`, `NK`, `NM`).

The current `adam_assist` handoff ignores those fixed metadata fields and uses the supported estimated `A2` value.

In [ ]:
orbit_54509 = load_sbdb_orbit("54509.json", object_id="54509")
attempt_supported_case("SBDB 54509", "54509", orbit_54509, day_offsets)


## Case 3: SBDB `101955`

This is a useful boundary case. The fixture carries an SRP-style solution that
estimates `AMRAT` and `RHO` alongside `A1/A2/A3`.

Ingestion keeps the supported `A1/A2/A3` values and their covariance rows, drops
the `AMRAT`/`RHO` values with a warning, and marginalizes their dimensions out of
the covariance. Propagation then uses the supported accelerations only — note
that this is an approximation for objects whose fitted dynamics included the
dropped parameters.


In [ ]:
orbit_101955 = load_sbdb_orbit("101955_phys.json", object_id="101955")
display(nongrav_summary(orbit_101955))

propagated_101955 = propagator.propagate_orbits(
    orbit_101955, future_times(orbit_101955, np.array([365.0]))
)
display(propagated_101955.coordinates.values)


## Case 4: SBDB `67P`

This comet fixture includes estimated `A1/A2/A3` plus `DT`.

Ingestion keeps `A1/A2/A3` and drops `DT` with a warning (its covariance
dimension is marginalized out). Propagation uses the standard Marsden `g(r)`
with the stored `A1/A2/A3` — again an approximation for a solution that was
fit with a time delay.


In [ ]:
orbit_67p = load_sbdb_orbit("67P_phys.json", object_id="67P")
display(nongrav_summary(orbit_67p))

propagated_67p = propagator.propagate_orbits(
    orbit_67p, future_times(orbit_67p, np.array([365.0]))
)
display(propagated_67p.coordinates.values)


## Case 5: NEOCC `99942`

This case is useful because the NEOCC fixture stores a supported `A2` together
with a non-estimated `AMRAT = 0` entry.

Ingestion keeps the useful `A2` (converted to canonical au/d^2) and drops the
`AMRAT` metadata; the `A2` uncertainty rides along in the extended coordinate
covariance.


In [ ]:
neocc_data_99942, neocc_nongrav_99942 = load_neocc_nongrav("99942.ke1")
display(pd.DataFrame({key: [value] for key, value in neocc_data_99942["nongrav"].items()}))
display(
    pd.DataFrame(
        {
            "source": neocc_nongrav_99942.source.to_pylist(),
            "A1": neocc_nongrav_99942.A1.to_pylist(),
            "A2": neocc_nongrav_99942.A2.to_pylist(),
            "A3": neocc_nongrav_99942.A3.to_pylist(),
        }
    )
)

synthetic_orbit = Orbits.from_kwargs(
    orbit_id=["99942-neocc-demo"],
    object_id=["99942"],
    non_gravitational_parameters=neocc_nongrav_99942,
    coordinates=orbit_99942.coordinates,
)
print("Particle params passed to ASSIST:", _extract_assist_particle_params(synthetic_orbit))


## How To Add Non-Gravs To An Orbit

At minimum, the current ASSIST path needs the canonical `A1/A2/A3` fields on `orbits.non_gravitational_parameters`.

A minimal pattern looks like this:

```python
from adam_core.orbits import Orbits
from adam_core.orbits.non_gravitational_parameters import NonGravitationalParameters

orbits = existing_orbits.set_column(
    "non_gravitational_parameters",
    NonGravitationalParameters.from_kwargs(
        source=["manual"],
        A1=[None],
        A2=[-8.7e-14],
        A3=[None],
    ),
)
```

Then propagate with:

```python
from adam_assist.propagator import ASSISTPropagator
propagator = ASSISTPropagator()
propagated = propagator.propagate_orbits(orbits, times)
```

If the estimated parameters include unsupported solved terms like `DT`, `AMRAT`, or `RHO`, the current runtime should raise a clear error.

## Suggested Real-World Review Order

If we want to validate correctness together after this notebook:

1. Start with `99942` and `54509` because they exercise the supported asteroid path and show measurable trajectory differences.
2. Compare the optional Horizons residuals when the environment has network access.
3. Inspect `101955` to confirm the current SRP boundary is enforced the way we expect.
4. Inspect `67P` to confirm the current cometary `DT` boundary is also explicit.

That should give us a concrete basis for deciding whether the next work should be model expansion, covariance generalization, or tighter external validation.